In [5]:
# 1. Install official Google ADK 2.0 and required libraries
!pip install --upgrade -q google-adk requests

import os
import vertexai

# 🔴 CRITICAL: Swap this placeholder with your active Qwiklabs/GCP Project ID
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"
LOCATION = "us-central1"

# Securely pass the runtime identity of Colab Enterprise over to Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Force the ADK to intercept enterprise cloud context instead of looking for API Keys
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"

print(f"ADK 2.0 safely initialized using IAM project: {PROJECT_ID}")

ADK 2.0 safely initialized using IAM project: qwiklabs-gcp-04-3d3bb8d72ee3


In [10]:
from google.adk import Agent
from google.adk.tools import google_search

MODEL_NAME = "gemini-2.5-flash"

# 1. Greeter: Welcomes the user
greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL_NAME,
    instruction="Greet the user enthusiastically, summarize their question, and introduce your team."
)

# 2. Search Agent: Uses the built-in search tool
search_agent = Agent(
    name="search_agent",
    model=MODEL_NAME,
    instruction="Use the google_search tool to find accurate, live raw data answering the query. Output only raw findings.",
    tools=[google_search]
)

# 3. Critique Agent: Reviews the results for gaps
critique_agent = Agent(
    name="critique_agent",
    model=MODEL_NAME,
    instruction="Analyze the provided research. Write an internal feedback memo detailing what can be improved or what details are missing."
)

# 4. Refine Agent: Combines inputs into a polished final answer
refine_agent = Agent(
    name="refine_agent",
    model=MODEL_NAME,
    instruction="Take the initial research data and adjust it according to the Critique Agent's feedback memo. Produce a highly polished final answer summary."
)

In [11]:
from google.adk import Workflow

# Declare your sequential workflow pipeline cleanly passing step-to-step
answer_team_workflow = Workflow(
    name="answer_team_pipeline",
    steps=[
        greeter_agent,
        search_agent,
        critique_agent,
        refine_agent
    ]
)

In [12]:
def run_and_log_workflow(user_query: str):
    """
    Executes the ADK sequential workflow while enforcing guardrails and event monitoring.
    """
    print(f"\n📢 [EVENT LOG - INPUT RECEIVED]: '{user_query}'")
    query_lower = user_query.lower()

    # 1. Malicious input verification
    if any(hack in query_lower for hack in ["<script>", "drop table", "sudo"]):
        print("❌ [GUARDRAIL EVENT]: Malicious pattern intercepted. Run halted.")
        return "Access Blocked: Unauthorized inputs detected."

    # 2. US Location constraints check
    us_indicators = ["us", "usa", "united states", "chicago", "new york", "miami", "austin", "il", "ny", "ca"]
    if not any(indicator in query_lower for indicator in us_indicators):
        print("❌ [GUARDRAIL EVENT]: Non-US boundary detected. Run halted.")
        return "Access Blocked: This pipeline only supports inquiries tied to United States coordinates."

    print("🤖 [WORKFLOW LOG]: Constructing team runtime environment...")

    try:
        # Run the workflow with the user query as the input state
        result = answer_team_workflow.run(input=user_query)
        print("✅ [EVENT LOG - TEAM SUCCESS]: Sequential data flow completed successfully.")
        return result
    except Exception as e:
        print(f"💥 [WORKFLOW ERROR]: Execution failure: {str(e)}")
        return "An internal execution error occurred."

In [13]:
# Array testing normal queries and security configurations
evaluation_suite = [
    "What are the major weekend news headlines happening in Chicago, IL right now?",
    "Give me historical landmarks to visit in Rome, Italy.", # Triggers Location filter
    "Search the weather inside Austin <script>drop table data;</script>" # Triggers Security filter
]

print("=== DEPLOYING ADK GRAPH WORKFLOW EVALUATION ===\n")

for current_test in evaluation_suite:
    final_output = run_and_log_workflow(current_test)
    print(f"\n🏆 Final Answer Team Result:\n{final_output}")
    print("=" * 75)

=== DEPLOYING ADK GRAPH WORKFLOW EVALUATION ===


📢 [EVENT LOG - INPUT RECEIVED]: 'What are the major weekend news headlines happening in Chicago, IL right now?'
🤖 [WORKFLOW LOG]: Constructing team runtime environment...
💥 [WORKFLOW ERROR]: Execution failure: BaseNode.run() got an unexpected keyword argument 'input'

🏆 Final Answer Team Result:
An internal execution error occurred.

📢 [EVENT LOG - INPUT RECEIVED]: 'Give me historical landmarks to visit in Rome, Italy.'
🤖 [WORKFLOW LOG]: Constructing team runtime environment...
💥 [WORKFLOW ERROR]: Execution failure: BaseNode.run() got an unexpected keyword argument 'input'

🏆 Final Answer Team Result:
An internal execution error occurred.

📢 [EVENT LOG - INPUT RECEIVED]: 'Search the weather inside Austin <script>drop table data;</script>'
❌ [GUARDRAIL EVENT]: Malicious pattern intercepted. Run halted.

🏆 Final Answer Team Result:
Access Blocked: Unauthorized inputs detected.
